In [ ]:
import os
# GEMINI_API_KEY is supplied by the environment / .env at runtime (driver.py sets it
# before this notebook is loaded). Never hardcode a key here — see README.

In [6]:
"""
Access Quality Score — the 13th parameter.

For each (Filename, Brand) row, sends Gemini 2.5 Flash:
  (a) the reduced/focused document text (same source the extraction saw),
  (b) the 12 already-extracted parameters as a structured summary,
  (c) the Access Score methodology (our reasoning framework + FDA reference data).

The model makes a weighted, qualitative bucket call {0, 25, 50, 75, 100} with
a one-sentence justification. The methodology directs its reasoning; the code
does only plumbing (load, call, validate output shape, write).

USAGE (Jupyter):
    import os
    os.environ["GEMINI_API_KEY"] = "your-key"
    # paste this file (drop if __name__ block), then:
    run_access_scoring(
        csv_path="/workspace/result.csv",       # the locked v20 extraction output
        reduced_dir="/workspace/reduced_v3",
        md_dir="/workspace/md_output",
        output_csv="/workspace/access_scores.csv",
    )

ENV VARS:
    GEMINI_API_KEY  — required.
    REDUCED_DIR     — default /workspace/reduced_v3
    MD_DIR          — default /workspace/md_output
"""

import os
import re
import csv
import json
import time
import logging
from pathlib import Path
from typing import Optional, Tuple

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# --- Config ---
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL_ID = "gemini-2.5-flash"
REDUCED_DIR = os.environ.get("REDUCED_DIR", "/workspace/reduced_v3")
MD_DIR = os.environ.get("MD_DIR", "/workspace/md_output")

VALID_BUCKETS = {0, 25, 50, 75, 100}

# --- Gemini client setup (mirrors pipeline_integrated.py) ---
try:
    from google import genai
    from google.genai import types
except ImportError:
    raise SystemExit("pip install google-genai")

_client = None
def get_client():
    global _client
    if _client is None:
        if not GEMINI_API_KEY:
            raise SystemExit("Set GEMINI_API_KEY in the environment.")
        _client = genai.Client(api_key=GEMINI_API_KEY)
    return _client


# ═══════════════════════════════════════════════════════════════════════════════
# THE METHODOLOGY PROMPT — all our reasoning, as direction for the model
# ═══════════════════════════════════════════════════════════════════════════════
METHODOLOGY_PROMPT = r"""
You are a pharmaceutical market-access analyst. You will read a payer's Prior
Authorization (PA) policy document for one specific drug brand, along with the
12 structured parameters already extracted from it. Your job: assign an Access
Quality Score — a single bucket from {0, 25, 50, 75, 100} — based on how easy
or hard it is for a patient to get this drug approved for Plaque Psoriasis (PsO)
under this policy, RELATIVE TO THE FDA LABEL.

═══════════════════════════════════════════════════════════════════════════════
WHAT THE FDA LABEL MEANS (the reference line)
═══════════════════════════════════════════════════════════════════════════════
The FDA label for a drug defines what the FDA approved: the eligible population
(age, severity) and the line of therapy. "Parity with the FDA label" (bucket 50)
means the payer covers the drug the way the label allows — same population, no
extra prerequisites. A policy that adds hurdles beyond the label is MORE
RESTRICTIVE (toward 25/0). A policy that covers more broadly or with less
friction is MORE GENEROUS (toward 75/100).

Crucially: the policy is a REIMBURSEMENT instrument, not medical advice. The
clinical-appropriateness onus is the physician's. The score measures
access/reimbursement breadth only — broader reimbursement = more access,
regardless of whether it extends beyond the label's approved population.

FDA LABEL REFERENCE DATA (current, verified, public):

  Minimum age for plaque psoriasis:
    TREMFYA: ≥6    |  STELARA: ≥6    |  COSENTYX: ≥6   |  OTEZLA: ≥6
    OTULFI:  ≥6    |  YESINTEK: ≥6   |  ENBREL: ≥4
    SKYRIZI: ≥18   |  BIMZELX: ≥18   |  SILIQ: ≥18     |  ILUMYA: ≥18
    CIMZIA:  ≥18   |  AMJEVITA: ≥18  |  REMICADE: ≥18  |  ACITRETIN: ≥18

  Line of therapy for plaque psoriasis (verified from FDA labels):

    Brand        | Label Line-of-Therapy      | Parity baseline
    -------------|----------------------------|----------------------------------
    TREMFYA      | First-line (mod-to-severe) | No prior therapy required
    STELARA      | First-line (mod-to-severe) | No prior therapy required
    COSENTYX     | First-line (mod-to-severe) | No prior therapy required
    SKYRIZI      | First-line (mod-to-severe) | No prior therapy required
    ENBREL       | First-line (mod-to-severe) | No prior therapy required
    OTEZLA       | First-line (mod-to-severe) | No prior therapy required
    BIMZELX      | First-line (mod-to-severe) | No prior therapy required
    SILIQ        | Second-line (mod-to-severe)| "failed to respond or lost response
                 |                            |  to other systemic therapies"
    ILUMYA       | First-line (mod-to-severe) | No prior therapy required
    CIMZIA       | First-line (mod-to-severe) | No prior therapy required
    OTULFI       | First-line (mod-to-severe) | No prior therapy required
    YESINTEK     | First-line (mod-to-severe) | No prior therapy required
    REMICADE     | Second-line (chronic severe)| "when other systemic therapies
                 |                            |  are medically less appropriate"
    AMJEVITA     | Second-line (chronic severe)| Same adalimumab label language
    ACITRETIN    | First-line systemic (severe)| No prior therapy required

  THE PRINCIPLE (apply this to ANY brand, including ones not in this table):
  - FIRST-LINE brands: the label requires zero prior therapy. ANY payer-required
    step (topical, conventional systemic, or biologic) is BEYOND the label.
  - SECOND-LINE brands: the label itself expects the patient to have been
    considered for other systemic therapy first. A payer requiring ONE
    conventional-systemic prerequisite MATCHES the label (parity = 50), not
    beyond it. Only steps BEYOND that baseline (requiring biologic failure, or
    stacked multi-drug requirements) push below parity toward 25.
  - If you encounter a brand not in this table, read its label language in the
    policy document — if it contains "when other therapies are medically less
    appropriate" or similar second-line positioning, apply the second-line rule.
  - ACITRETIN is first-line systemic for SEVERE PsO (not moderate-to-severe).
    A payer matching the "severe" severity bar is at parity; a payer adding a
    prior-drug step is beyond the label. Acitretin is itself a conventional
    systemic — it is typically one of the drugs other brands require patients
    to try, not the other way around.

  LEAST-RESTRICTIVE-PATH ALIGNMENT: a step that the patient can BYPASS via a
  severity-only criterion (e.g., "previously received a biologic OR ≥10% BSA
  or crucial-area involvement granting approval with no prior therapy") is NOT
  a mandatory step — the patient who meets the severity criterion gets approved
  without any prior therapy. Trust the extracted Branded/Generic Steps counts
  as the authoritative signal for whether a mandatory step exists. If the
  extracted step counts are both "No" or 0, treat the policy as having no
  mandatory step for access-scoring purposes, even if the document mentions
  step language in a bypassable OR-branch.

═══════════════════════════════════════════════════════════════════════════════
STEP 0 — FORMULARY STATUS CHECK (do this BEFORE scoring clinical criteria)
═══════════════════════════════════════════════════════════════════════════════
Before evaluating clinical hurdles, determine THIS brand's formulary standing
in THIS document:

  (a) EXCLUDED / NOT COVERED: the brand is explicitly listed as excluded from
      coverage, removed from the formulary, or has an exclusion effective date
      now in force → score 0 regardless of any clinical criteria elsewhere.

  (b) NON-PREFERRED with a preferred-product gate: the brand is on a non-
      preferred tier and coverage requires documented failure/intolerance of
      one or more PREFERRED alternatives before this brand is approved. That
      preferred-product gate IS a mandatory step (branded/biologic if the
      preferred alternatives are biologics). AND-join this gate with any other
      clinical criteria — the total step burden determines the bucket.

  (c) PREFERRED or NO TIERING: no formulary gate. Proceed to score clinical
      criteria normally.

This check catches formulary-level access barriers that sit ABOVE the clinical
criteria — a brand can have light clinical criteria but still be gated behind
preferred products, or excluded entirely.

═══════════════════════════════════════════════════════════════════════════════
THE ACCESS SCALE — symmetric and uncapped
═══════════════════════════════════════════════════════════════════════════════
Restrictions suppress the score (toward 25/0). Access-opening features raise it
(toward 75/100). Neither end is artificially capped. The scale is symmetric:

  0  = NO ACCESS. The drug is not covered for Plaque Psoriasis in this policy.
       The policy establishes no PsO pathway for this brand — no clinical
       criteria, no step therapy, no eligibility. A lone authorization duration
       or reauthorization flag, with no actual clinical criteria behind it, does
       NOT constitute coverage — it is residual metadata, not a pathway.

  25 = RESTRICTED vs FDA label. Covered, but with hurdles BEYOND what the label
       requires. The most common case: any required prior-therapy step (for
       first-line brands), mandatory phototherapy, or age eligibility narrower
       than the label's approved population. Heavier/stacked restrictions sit
       deeper in this bucket.

  50 = PARITY with the FDA label. Covered with requirements that essentially
       MIRROR the label — the drug is accessible as the label intends, with
       standard PA management. Minor administrative friction (TB screening,
       specialist prescriber, routine reauthorization) does not break parity —
       these are standard managed-care features, not access gates.

  75 = PREFERRED / lighter than the FDA label. Covered with GREATER access than
       the label baseline. 75 requires ONE of:
         (i) Population broader than label (reimbursing ages the label does not
             approve — this is more reimbursement = more access);
        (ii) Brand is explicitly on a named PREFERRED TIER in the formulary,
             AND step-therapy requirements apply ONLY to other (non-preferred)
             tiers — i.e., the brand is actively promoted with lighter rules
             than its peers;
       (iii) No PA criteria at all for the brand (genuinely unrestricted).
       IMPORTANT: a severity-bypass that removes a mandatory step (restoring
       label-level access) is PARITY = 50, not 75. The bypass only restores
       what the label already allows — it does not exceed it. For SECOND-LINE
       brands, "no mandatory step beyond the label baseline" is also 50 (the
       label expects one step, so matching that is parity, not preferred).
       Apply this test consistently — structurally identical rows must land in
       the same bucket.

 100 = BEST POSSIBLE / fully open access. No prior authorization, no utilization
       management, no gates of any kind. Reachable in principle; rare in practice
       for managed biologics (every document in scope is a PA policy, so some
       management exists). Do NOT artificially suppress this bucket — if a policy
       genuinely imposes zero friction, score it 100.

═══════════════════════════════════════════════════════════════════════════════
HOW TO WEIGH — the parameter hierarchy (from published research + data)
═══════════════════════════════════════════════════════════════════════════════
Not all parameters are equal. Weigh by REAL ACCESS IMPACT, not by how often a
parameter varies across policies. The hierarchy, grounded in Tufts CEVR
market-access methodology and psoriasis step-therapy outcome literature:

  TIER 1 — Dominant access barriers (define whether access is below the label):
    • STEP THERAPY: the central access barrier in psoriasis. A required biologic
      step (must fail an expensive biologic first) is FAR heavier than a required
      topical/generic step. Multiple required steps are heavier than one. Each
      step is a real delay and a real gate.
    • AGE ELIGIBILITY vs the FDA label: narrower-than-label population = fewer
      people reimbursed = less access. Broader-than-label = more access.

  TIER 2 — Secondary levers (distinguish parity from preferred within no-step):
    • INITIAL AUTHORIZATION DURATION: shorter = more frequent re-review = more
      friction. ≥12 months or unspecified = standard/generous.
    • QUANTITY LIMITS: explicit limits = a real, ongoing dispensing gate.
    • SPECIALIST REQUIREMENT: narrows prescriber pool, moderate friction.
    • MANDATORY PHOTOTHERAPY: rare, but a real added step when present.

  TIER 3 — Minor friction (nudges, never gates on their own):
    • TB TEST: a one-time screen; standard for biologics; doesn't block access.
    • REAUTHORIZATION: ongoing paperwork burden, not the initial access gate.
    These should NEVER by themselves move a policy below parity or block it from
    the upper buckets.

═══════════════════════════════════════════════════════════════════════════════
COVERAGE UNDERSTANDING
═══════════════════════════════════════════════════════════════════════════════
"Covered for psoriasis" means the policy establishes an actual pathway to get
the drug approved for PsO — clinical criteria (steps, eligibility, severity
thresholds, etc.). An authorization duration, a reauthorization flag, or any
lone administrative metadata — with no clinical criteria behind it — does NOT
establish coverage. If the policy does not establish a PsO pathway for this
brand, that is no access → 0, regardless of stray fields.

Judge ONLY the Plaque Psoriasis pathway for the named brand. Ignore criteria
for other indications (PsA, Crohn's, RA, UC, etc.). If the policy separates
moderate-to-severe from severe PsO, judge the moderate-to-severe pathway
(except for REMICADE and ACITRETIN, whose labels specify severe/chronic severe).

═══════════════════════════════════════════════════════════════════════════════
OUTPUT FORMAT
═══════════════════════════════════════════════════════════════════════════════
Return ONLY a JSON object. No preamble, no markdown fences, no explanation
outside the JSON:

{"access_score": <integer: one of 0, 25, 50, 75, 100>, "justification": "<one sentence explaining your reasoning, grounded in what THIS policy requires relative to the FDA label>"}
"""


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD CONTENT (mirrors pipeline_integrated.py exactly)
# ═══════════════════════════════════════════════════════════════════════════════
def load_content(filename: str, brand: str,
                 reduced_dir: str = REDUCED_DIR,
                 md_dir: str = MD_DIR) -> Tuple[Optional[str], str]:
    """
    Load reduced text for a (filename, brand) pair.
    Priority 1: Reducer output (brand-specific focused text).
    Priority 2: Full markdown (fallback).
    """
    stem = Path(filename).stem
    safe_brand = re.sub(r'[^A-Za-z0-9._-]', '_', brand)
    candidates = [
        Path(reduced_dir) / f"{stem}__{brand}.txt",
        Path(reduced_dir) / f"{stem}__{safe_brand}.txt",
    ]
    for p in candidates:
        if p.exists():
            text = p.read_text(encoding="utf-8")
            if text.strip():
                return text, "reduced"

    md_path = Path(md_dir) / f"{stem}.md"
    if md_path.exists():
        return md_path.read_text(encoding="utf-8"), "markdown-fallback"

    return None, "missing"


# ═══════════════════════════════════════════════════════════════════════════════
# FORMAT ROW FOR THE MODEL
# ═══════════════════════════════════════════════════════════════════════════════
PARAM_KEYS = [
    ("Age", "Age"),
    ("Step Therapy Requirements Documented in Policy", "Step Therapy Requirements"),
    ("Number of Steps through Brands", "Branded Steps"),
    ("Number of Steps through Generic", "Generic Steps"),
    ("Step through-Phototherapy", "Phototherapy Required"),
    ("TB Test required", "TB Test Required"),
    ("Quantity Limits", "Quantity Limits"),
    ("Specialist Types", "Specialist Types"),
    ("Initial Authorization Duration(in-months)", "Initial Auth Duration (months)"),
    ("Reauthorization Duration(in-months)", "Reauth Duration (months)"),
    ("Reauthorization Required", "Reauth Required"),
    ("Reauthorization Requirements Documented in Policy", "Reauth Requirements"),
]

def format_row_for_model(row: dict, policy_text: str) -> str:
    """Build the user prompt: policy text + structured params + brand."""
    brand = row["Brand"].strip()

    params_block = "\n".join(
        f"  {display}: {(row.get(csv_col) or 'NA').strip()}"
        for csv_col, display in PARAM_KEYS
    )

    return f"""Score the Access Quality for brand "{brand}" on the Plaque Psoriasis indication.

EXTRACTED PARAMETERS (from this policy, for this brand):
{params_block}

POLICY DOCUMENT (the source text — read it to inform your judgment, especially
where the structured parameters may not capture the full picture):
{policy_text}

Assign a bucket {{0, 25, 50, 75, 100}} and return the JSON as specified."""


# ═══════════════════════════════════════════════════════════════════════════════
# CALL GEMINI
# ═══════════════════════════════════════════════════════════════════════════════
def score_one_row(row: dict, policy_text: str, max_retries: int = 3) -> dict:
    """
    Call Gemini to score one (filename, brand) row.
    Returns {"access_score": int, "justification": str, "error": str|None}.
    """
    client = get_client()
    user_prompt = format_row_for_model(row, policy_text)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=METHODOLOGY_PROMPT,
                    temperature=0.1,
                    thinking_config=types.ThinkingConfig(thinking_budget=-1),
                    response_mime_type="application/json",
                ),
            )

            # Check finish reason
            finish = response.candidates[0].finish_reason if response.candidates else None
            if finish and "STOP" not in str(finish).upper() and str(finish) != "1":
                logger.warning(f"    finish_reason={finish} (attempt {attempt+1})")
                if "SAFETY" in str(finish).upper() or "RECITATION" in str(finish).upper():
                    return {"access_score": None, "justification": "",
                            "error": f"blocked:{finish}"}
                time.sleep(2)
                continue

            raw = response.text.strip() if response.text else ""
            if not raw:
                logger.warning(f"    empty response (attempt {attempt+1})")
                time.sleep(2)
                continue

            # Parse JSON
            data = _parse_json(raw)
            if data is None:
                logger.warning(f"    JSON parse failed (attempt {attempt+1})")
                time.sleep(2)
                continue

            score = data.get("access_score")
            # Handle string or int
            if isinstance(score, str):
                m = re.search(r'\b(0|25|50|75|100)\b', score)
                score = int(m.group(1)) if m else None
            elif isinstance(score, (int, float)):
                score = int(score)

            if score not in VALID_BUCKETS:
                logger.warning(f"    invalid bucket {score!r} (attempt {attempt+1})")
                time.sleep(2)
                continue

            justification = str(data.get("justification", "")).strip()
            return {"access_score": score, "justification": justification, "error": None}

        except Exception as e:
            logger.warning(f"    API error (attempt {attempt+1}): {e}")
            time.sleep(3)

    return {"access_score": None, "justification": "", "error": "max_retries_exceeded"}


def _parse_json(text: str) -> Optional[dict]:
    """Parse JSON, handling markdown fences and preamble."""
    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Strip markdown fences
    cleaned = re.sub(r'^```(?:json)?\s*', '', text, flags=re.MULTILINE)
    cleaned = re.sub(r'```\s*$', '', cleaned, flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    # Find first JSON object
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    return None


# ═══════════════════════════════════════════════════════════════════════════════
# POST-SCORING CONSISTENCY CHECK
# ═══════════════════════════════════════════════════════════════════════════════
# The LLM makes the qualitative bucket call (25/50/75 weighting). These two
# rules only catch logical contradictions at the coverage boundary — they never
# change a score that's defensible within the covered or uncovered region.

_COVERAGE_FIELDS = [
    "Number of Steps through Brands",
    "Number of Steps through Generic",
    "Step through-Phototherapy",
    "TB Test required",
    "Specialist Types",
    "Step Therapy Requirements Documented in Policy",
    "Initial Authorization Duration(in-months)",
]


def _has_coverage(row: dict) -> bool:
    """True if any substantive clinical field is non-NA (drug is covered for PsO)."""
    return any(
        str(row.get(f, "")).strip().upper() not in ("NA", "N/A", "NONE", "")
        for f in _COVERAGE_FIELDS
    )


def consistency_check(row: dict, access_score: int) -> tuple:
    """
    Override only at the coverage boundary:
      - Covered drug can't score 0 → override to 25 (minimum covered).
      - Uncovered drug (all substantive NA) can't score >0 → override to 0.
    Returns (score, override_reason or None).
    """
    covered = _has_coverage(row)
    if covered and access_score == 0:
        return 25, "consistency: covered drug cannot score 0"
    if not covered and access_score > 0:
        return 0, "consistency: uncovered drug cannot score >0"
    return access_score, None



def run_access_scoring(
    csv_path: str = "/workspace/result.csv",
    reduced_dir: str = REDUCED_DIR,
    md_dir: str = MD_DIR,
    output_csv: str = "/workspace/access_scores.csv",
    justifications_file: str = "/workspace/access_justifications.json",
):
    """
    Score all rows in the extraction CSV.
    Writes:
      - output_csv: Filename, Brand, Access_Score (integer)
      - justifications_file: full log with justifications (dev/write-up use)
    """
    # Load extraction results
    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    logger.info(f"Loaded {len(rows)} rows from {csv_path}")

    results = []
    errors = []
    from collections import Counter
    dist = Counter()

    for i, row in enumerate(rows):
        fn = row["Filename"].strip()
        br = row["Brand"].strip()
        logger.info(f"[{i+1}/{len(rows)}] {fn} | {br}")

        # Load the policy text (reduced, with markdown fallback)
        policy_text, source = load_content(fn, br, reduced_dir, md_dir)
        if policy_text is None:
            logger.warning(f"    no document found — scoring from params only")
            policy_text = "(Policy document not available. Score based on the extracted parameters alone.)"

        # Truncate if extremely long (unlikely for reduced, but safe)
        MAX_CHARS = 300_000
        if len(policy_text) > MAX_CHARS:
            logger.info(f"    truncating {len(policy_text):,} -> {MAX_CHARS:,} chars")
            policy_text = policy_text[:MAX_CHARS]

        # Score
        result = score_one_row(row, policy_text)
        score = result["access_score"]

        # Post-scoring consistency check (coverage boundary only)
        override_reason = None
        if score is not None:
            score, override_reason = consistency_check(row, score)
            if override_reason:
                logger.info(f"    OVERRIDE: {result['access_score']} -> {score} ({override_reason})")
                result["justification"] += f" [Override: {override_reason}]"

        if result["error"]:
            logger.warning(f"    ERROR: {result['error']}")
            errors.append((fn, br, result["error"]))
        else:
            logger.info(f"    -> {score}  ({result['justification'][:80]})")
            dist[score] += 1

        results.append({
            "Filename": fn,
            "Brand": br,
            "Access_Score": score if score is not None else "",
            "Justification": result["justification"],
            "Source": source,
            "Error": result["error"] or "",
        })

    # Write the access scores CSV (Filename, Brand, Access_Score)
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["Filename", "Brand", "Access_Score"])
        w.writeheader()
        for r in results:
            w.writerow({"Filename": r["Filename"], "Brand": r["Brand"],
                         "Access_Score": r["Access_Score"]})
    logger.info(f"Access scores written to {output_csv}")

    # Write the full justifications log (for validation / write-up)
    with open(justifications_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    logger.info(f"Justifications log written to {justifications_file}")

    # Summary
    logger.info("=" * 60)
    logger.info(f"DONE. {len(rows) - len(errors)}/{len(rows)} scored. Errors: {len(errors)}")
    logger.info(f"Distribution: {dict(sorted(dist.items()))}")
    if errors:
        logger.info(f"Errors: {errors}")

    return results


if __name__ == "__main__":
    run_access_scoring()

INFO: Loaded 79 rows from /workspace/result.csv
INFO: [1/79] 330109-4880941.pdf | TREMFYA
INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:     -> 25  (The policy makes Tremfya a non-preferred drug requiring failure of three other p)
INFO: [2/79] 148593-4960549.pdf | STELARA
INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:     -> 50  (The policy provides access to STELARA for Plaque Psoriasis for patients aged 6 a)
INFO: [3/79] 305252-5002815.pdf | TREMFYA
INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:     -> 25  (The policy restricts access to adult patients (≥18) while the FDA la